# Lab 1 - Try Out Azure AI Services

**Day 1 - AI Fundamentals and Machine Learning** | Estimated time: **40 minutes**

**Azure AI services** (now branded *Foundry Tools*) are **pre-trained AI models behind an API**.
You don't train anything: you send data and get predictions back.

You will call four services through **one shared Azure resource**:

| Part | Service | You will... |
|---|---|---|
| 1 | Language (raw REST) | see exactly what an AI API call looks like on the wire |
| 2 | Language (Python SDK) | detect sentiment and opinions, key phrases, entities and **PII** |
| 3 | Vision | caption and tag an image, and read text from a receipt (OCR) |
| 4 | Translator | translate and transliterate Indian languages |
| 5 | All of them | build a multilingual customer-feedback pipeline |

> **Before you start:** your trainer has given you an **endpoint**, a **key** and a **region**.
> Put them in the `.env` file at the root of the lab folder (see the lab guide, Task 1).

In [ ]:
import json
from pathlib import Path

import pandas as pd
from lab_helpers import load_config, post_json, pretty, chunks, translator_endpoint

ENDPOINT, KEY, REGION = load_config()

If you see `Azure settings not found`, open `.env` in VS Code, paste the values from your trainer, **save**, and re-run the cell.

> **Security note:** the key is a password for a paid resource. Never paste it into code, chat, slides or a Git repository.
> Keeping it in `.env` (which is listed in `.gitignore`) is the minimum good practice.
> In production you would use **Microsoft Entra ID** (keyless) or **Azure Key Vault**.

---
## Part 1 - An AI service is just an HTTPS request (5 min)

Every Azure AI call has the same four ingredients:

1. **Endpoint URL**: which resource and which operation
2. **api-version**: which version of the API contract
3. **Header** `Ocp-Apim-Subscription-Key`: proves you're allowed to use it
4. **JSON body**: your data

In [ ]:
url = f"{ENDPOINT}/language/:analyze-text"
params = {"api-version": "2024-11-01"}
body = {
    "kind": "LanguageDetection",
    "analysisInput": {"documents": [
        {"id": "1", "text": "Hello, how are you today?"},
        {"id": "2", "text": "नमस्ते, आज आप कैसे हैं?"},
        {"id": "3", "text": "ನಮಸ್ಕಾರ, ಇವತ್ತು ಹೇಗಿದ್ದೀರಾ?"},
        {"id": "4", "text": "Bonjour, comment allez-vous aujourd'hui ?"},
        {"id": "5", "text": "Kal milte hain, office mein."},
    ]},
}

print("POST", url, params)
response = post_json(url, KEY, body, params=params)
pretty(response)

In [ ]:
# Pull out just what we need
for doc in response["results"]["documents"]:
    lang = doc["detectedLanguage"]
    text = next(d["text"] for d in body["analysisInput"]["documents"] if d["id"] == doc["id"])
    print(f"{lang['name']:<10} ({lang['iso6391Name']})  confidence {lang['confidenceScore']:.2f}  | {text}")

**What to notice**

* The response is plain JSON, so any language that can make an HTTPS call can use AI services.
* Document 5 is **Hinglish** (Hindi written in Latin script), and it's still detected as Hindi.
  The service identifies the *language*, not the *script*. Downstream steps such as translation may treat it differently.
* Each result comes with a **confidence score**. Real applications set a threshold for when to trust it.

---
## Part 2 - Language service with the Python SDK (10 min)

An **SDK** wraps the REST calls in typed Python objects, and handles retries and errors for you.

In [ ]:
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

language = TextAnalyticsClient(endpoint=ENDPOINT, credential=AzureKeyCredential(KEY))
print("Client ready:", type(language).__name__)

### 2a. Sentiment analysis with opinion mining

Sentiment is returned for the whole document **and** for each sentence.
**Opinion mining** goes further: it finds the *target* (for example "battery") and the *assessment* (for example "drains quickly").

In [ ]:
reviews = [
    "The hotel room was spotless and the staff were wonderful.",
    "The camera is excellent but the battery drains far too quickly.",
    "Worst customer service ever. I waited 45 minutes on hold.",
    "The package arrived on Tuesday.",
]

results = language.analyze_sentiment(reviews, show_opinion_mining=True)

for review, doc in zip(reviews, results):
    if doc.is_error:
        print("Error:", doc.error); continue
    s = doc.confidence_scores
    print(f"\n{doc.sentiment.upper():<8} (pos {s.positive:.2f} / neu {s.neutral:.2f} / neg {s.negative:.2f})  {review}")
    for sentence in doc.sentences:
        for opinion in sentence.mined_opinions:
            assessments = ", ".join(f"'{a.text}' ({a.sentiment})" for a in opinion.assessments)
            print(f"    target '{opinion.target.text}' -> {assessments}")

Look at the camera review. Its **overall** label is NEGATIVE, yet opinion mining finds *camera → excellent* (positive).
A single document label hides that nuance; opinion mining shows **what** customers like or dislike, which is far more actionable.
Notice also that it missed *battery → drains*. Pre-trained models are good, not perfect.

### 2b. Key phrases, named entities and entity linking

In [ ]:
article = ("Infosys announced on Monday that it will open a new AI research centre in Mysuru, Karnataka, "
           "investing 500 crore rupees over three years. CEO Salil Parekh said the centre will hire 2,000 engineers.")

kp = language.extract_key_phrases([article])[0]
print("Key phrases:", kp.key_phrases)

print("\nNamed entities:")
for e in language.recognize_entities([article])[0].entities:
    sub = f"/{e.subcategory}" if e.subcategory else ""
    print(f"  {e.text:<22} {e.category}{sub:<14} {e.confidence_score:.2f}")

print("\nLinked entities (disambiguated against Wikipedia):")
for e in language.recognize_linked_entities([article])[0].entities:
    print(f"  {e.name:<22} {e.url}")

### 2c. Detect and redact personal data (PII)

Before you store customer text, or send it to an LLM, you often must remove personal data.
Check the output carefully: does it redact anything that **isn't** personal data?

In [ ]:
ticket = ("Hi, I'm Anita Rao. My order 55821 never arrived. Email me at anita.rao@example.com "
          "or call +91 98860 11223. My card ending 4417 was charged twice.")

pii = language.recognize_pii_entities([ticket])[0]
print("Redacted:", pii.redacted_text, "\n")
for e in pii.entities:
    print(f"  {e.category:<16} {e.text!r:<30} {e.confidence_score:.2f}")

> **Try it (3 min)**
> 1. Write a review with sarcasm, for example *"Oh great, another update that breaks everything."* Is the sentiment right?
> 2. Add a review in Hindi to `reviews`. Does sentiment still work?
> 3. Pass `categories_filter=["Email"]` to `recognize_pii_entities` so that **only** emails are redacted.

In [ ]:
# Your code here

---
## Part 3 - Vision: describe an image and read text (8 min)

The **Image Analysis 4.0** API returns several *visual features* from a single call.

In [ ]:
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures

vision = ImageAnalysisClient(endpoint=ENDPOINT, credential=AzureKeyCredential(KEY))

IMAGE_URL = "https://learn.microsoft.com/azure/ai-services/computer-vision/media/quickstarts/presentation.png"
print("Analyzing:", IMAGE_URL)

features = [VisualFeatures.CAPTION, VisualFeatures.TAGS, VisualFeatures.OBJECTS, VisualFeatures.PEOPLE]
try:
    result = vision.analyze_from_url(image_url=IMAGE_URL, visual_features=features, gender_neutral_caption=True)
except Exception as err:
    # Captions only exist in some Azure regions; fall back without them
    print("Caption not available in this region - continuing without it.\n ", str(err)[:150])
    features.remove(VisualFeatures.CAPTION)
    result = vision.analyze_from_url(image_url=IMAGE_URL, visual_features=features)

if result.caption:
    print(f"\nCaption: '{result.caption.text}' (confidence {result.caption.confidence:.2f})")
print("\nTags   :", ", ".join(f"{t.name} ({t.confidence:.2f})" for t in result.tags.list[:10]))
print("Objects:", [(o.tags[0].name, o.bounding_box.as_dict()) for o in result.objects.list])
print("People :", len([p for p in result.people.list if p.confidence > 0.5]), "with confidence > 0.5")

Open the image URL in your browser and compare it with the caption and tags. How well did the model do?

### 3b. OCR: read a receipt from a local file

In [ ]:
image_bytes = Path("data/receipt.png").read_bytes()
ocr = vision.analyze(image_data=image_bytes, visual_features=[VisualFeatures.READ])

lines = [line.text for block in ocr.read.blocks for line in block.lines]
print(f"Read {len(lines)} lines:\n")
print("\n".join(lines))

In [ ]:
# OCR gives you text. Pulling out a specific field is your code's job (or an LLM's)
import re
full_text = "\n".join(lines)

# GSTIN format: 2 digits, 5 letters, 4 digits, 1 letter, 1 letter/digit, 'Z', 1 letter/digit
gstin = re.search(r"\b\d{2}[A-Z]{5}\d{4}[A-Z][A-Z\d]Z[A-Z\d]\b", full_text)
amounts = re.findall(r"\b\d+\.\d{2}\b", full_text)

print("GSTIN          :", gstin.group(0) if gstin else "not found")
print("Largest amount :", max(amounts, key=float) if amounts else "not found", " <- probably the total")

> Open `data/receipt.png` in VS Code to see the image. It is slightly rotated on purpose, and OCR copes with that.
>
> **Try it:** set `IMAGE_URL` to any public image URL (for example a photo of a street or a product) and re-run the analysis cell.

---
## Part 4 - Translator (5 min)

Translator works with the same multi-service key. Because the key belongs to a **regional** resource,
you must also send the `Ocp-Apim-Subscription-Region` header.

In [ ]:
TRANSLATOR = translator_endpoint()

text = "Your refund has been processed and will reach your account within 5 working days."
result = post_json(f"{TRANSLATOR}/translate", KEY, [{"Text": text}], region=REGION,
                   params={"api-version": "3.0", "to": ["hi", "kn", "ta", "fr", "ja"]})

print("Detected source:", result[0]["detectedLanguage"])
for t in result[0]["translations"]:
    print(f"  {t['to']:>3}: {t['text']}")

In [ ]:
# Transliteration: change the SCRIPT, not the language (Devanagari -> Latin)
hindi = next(t["text"] for t in result[0]["translations"] if t["to"] == "hi")
translit = post_json(f"{TRANSLATOR}/transliterate", KEY, [{"Text": hindi}], region=REGION,
                     params={"api-version": "3.0", "language": "hi", "fromScript": "Deva", "toScript": "Latn"})
print("Hindi       :", hindi)
print("Romanised   :", translit[0]["text"])

---
## Part 5 - Put it together: a multilingual feedback pipeline (7 min)

A support team receives feedback in many languages. For each message we want the **language**, an
**English version**, its **sentiment**, its **key phrases**, and a **PII-free** copy that is safe to store.

```
feedback ──► Translator (detect + translate to EN) ──► Language: sentiment + key phrases + PII redaction ──► table
```

In [ ]:
feedback = json.loads(Path("data/customer_feedback.json").read_text(encoding="utf-8"))
print(len(feedback), "messages loaded")

# 1) Translate everything to English in ONE call (Translator accepts up to 1,000 texts per request)
tr = post_json(f"{TRANSLATOR}/translate", KEY, [{"Text": f["text"]} for f in feedback],
               region=REGION, params={"api-version": "3.0", "to": "en"})
for f, t in zip(feedback, tr):
    f["language"] = t["detectedLanguage"]["language"]
    f["english"] = t["translations"][0]["text"]

# 2) Language analysis on the English text, in batches of 5 (the synchronous API limit for PII)
english = [f["english"] for f in feedback]
sentiments, phrases, redacted = [], [], []
for batch in chunks(english, 5):
    sentiments += language.analyze_sentiment(batch)
    phrases += language.extract_key_phrases(batch)
    redacted += language.recognize_pii_entities(batch)

for f, s, k, r in zip(feedback, sentiments, phrases, redacted):
    f["sentiment"] = s.sentiment
    f["negative_score"] = round(s.confidence_scores.negative, 2)
    f["key_phrases"] = ", ".join(k.key_phrases[:4])
    f["safe_text"] = r.redacted_text

report = pd.DataFrame(feedback)[["id", "channel", "language", "sentiment", "negative_score", "key_phrases", "safe_text"]]
pd.set_option("display.max_colwidth", 80)
report

In [ ]:
# 3) Act on it: which messages need a human to call the customer back?
urgent = report[(report.sentiment == "negative") & (report.negative_score > 0.7)]
print(f"{len(urgent)} message(s) need a follow-up today:\n")
for _, row in urgent.iterrows():
    print(f"  {row.id} [{row.language}] {row.safe_text}")

**Discuss**

* You built a useful multilingual analytics pipeline **without training a single model**.
  That's the value of pre-trained AI services.
* These services are **predictive** AI: each returns a fixed kind of answer (a label, entities, a translation).
  Tomorrow's **generative** AI can do open-ended tasks, but costs more, is less predictable, and can hallucinate.
* We translated first, then analysed in English. What could be lost in translation?
  When would you analyse the original language directly?
* Look at `safe_text`: the default PII settings also masked words like *"staff"* and *"three days late"*.
  Over-redaction destroys useful information. In production you choose the categories with `categories_filter`.

> **Stretch:** add a column `needs_callback` that is `True` when the original text contains a phone number
> (hint: `recognize_pii_entities` returns entities with `category == "PhoneNumber"`).

---
## Summary

* An Azure AI service call = **endpoint + api-version + key header + JSON body**.
* SDKs (`azure-ai-textanalytics`, `azure-ai-vision-imageanalysis`) give typed results and built-in retries.
* One multi-service resource covers Language, Vision and Translator. Translator also needs the **region** header.
* Pre-trained services return **confidence scores**, so your application decides what to trust.
* Keep keys out of code. Redact PII before you store text or send it to other systems.